In [1]:
import matplotlib.pyplot as plt
import numpy as np
import json
import pandas as pd
import math



In [2]:
# CONVERTS DICTIONARIES TAKEN FROM .JSON FILES INTO DATAFRAMES FOR THE KEY EVENTS
def dict_key_conversion(data):
    temp_df = pd.DataFrame(columns=['test_number', 'dwell_time', 'flight_time', 'key_pressed'])
    temp_flight_df = pd.DataFrame(columns=['test_number', 'flight_time', 'key_released'])

    temp_df_count = 0 # indicates which row of the df the next row of data should be appeneded into
    temp_flight_df_count = 0

    for i in range(1, 11): # loops through each of the tests in true_data
        k_data = data['test_'+str(i)]['key_events']
        # removes tabs from the data, as kivy, which is the library used for data collection, doesn't register tab releases, only presses
        tabless_k_data = []
        for k in k_data:
            if k['Key'] != 'tab':
                tabless_k_data.append(k)

        count = 0 #counter for how many iterations into the for loop it is
        f_count = 0 #counter for how many iterations into the loop the flight section has done
        prev_key_press = 0
        prev_key_release = 0
        for j in tabless_k_data:
            if j['Event'] == 'pressed': # THIS EXECUTES TO FIND THE DWELL TIME
                flight_impute = 0.0 # imputes flight time as 0 for now, as there are instances of key presses not having releases at the end of the test
                key_id = j['Key'] # this is what the actual key that is being pressed/released is
                key_press_time = j['Epoch'] # the epoch time of the key press
                key_release = False # is true when the release of the key has been found
                cont_count = 1 # keeps track of counting from the current key press, as it loops from 

                while key_release == False: # continues 
                    c = cont_count + count
                    start_row = tabless_k_data[count]
                    next_row = tabless_k_data[c]
                    # executes if the row is the release of the key that was pressed, and exits the while loop
                    if next_row['Key'] == key_id and next_row['Event'] == 'released':
                        key_release_time = next_row['Epoch']
                        dwell_time = float(key_release_time) - float(key_press_time)
                        key_release = True
                    # executes if the next row is a press event for a different key
                    elif next_row['Key'] != key_id and next_row['Event'] == 'pressed':
                        cont_count += 1
                    elif next_row['Key'] != key_id and next_row['Event'] == 'released':
                        cont_count += 1
                    else:
                        key_release = True
                        dwell_time = 0
                        key_release_time = start_row['Epoch']

                temp_df.loc[temp_df_count] = [i, dwell_time, flight_impute, key_id]

                prev_key_press = key_press_time
                prev_key_release = key_release_time
                temp_df_count += 1

            count += 1

            if j['Event'] == 'released': # THIS EXECUTES TO FIND THE FLIGHT TIME
                key_id = j['Key']
                f_cont_count = 1
                flight_time = []
                flight_found = False
                while flight_found == False:
                    f_c = f_count + f_cont_count
                    if f_c < len(tabless_k_data):
                        next_row = tabless_k_data[f_c]
                        if next_row['Event'] == 'pressed' and next_row['Key'] != key_id:
                            flight_time = float(next_row['Epoch']) - float(j['Epoch'])
                            temp_flight_df.loc[temp_flight_df_count] = [i, flight_time, key_id]
                            temp_flight_df_count += 1
                            flight_found = True
                        f_cont_count += 1
                    else:
                        flight_found = True
            f_count += 1

    # Now merges the flight time df with the rest of the features
    for i in range(1, 11):
        fh_count = 0
        flight_hold = []
        for j in temp_flight_df.index:
            if temp_flight_df.at[j, 'test_number'] == i:
                flight_hold.append(temp_flight_df.at[j, 'flight_time'])
        fh_count = 0

        for j in temp_df.index:
            if temp_df.at[j, 'test_number'] == i and fh_count < len(flight_hold):
                temp_df.at[j, 'flight_time'] = flight_hold[fh_count]
                fh_count += 1

    true_k_df = temp_df
    return true_k_df

In [3]:
# CONVERTS DICTIONARIES TAKEN FROM .JSON FILES INTO DATAFRAMES FOR THE MOUSE EVENTS
def get_distance(a, b): # method used to calculate distance between two coordinates
    distance = math.sqrt(((a[0] - b[0]) ** 2) + ((a[1] - b[1]) ** 2))
    return distance

def dict_mouse_conversion(data):
    m_df = pd.DataFrame(columns = ['test_number', 'movement_id', 'trajectory', 'single_coor'])
    row_count = 0
    for i in range(1, 11):
        m_data = data['test_'+str(i)]['mouse_events']
        m_movements = []
        for j in m_data[:len(m_data)-1]:
            if j['Event'] == 'movement':
                m_movements.append(j)
        
        # creates dictionary that passes all the movement coordinates to the each movement ID in the test
        movement_coor_dict = {}
        for j in m_movements:
            movement_coor_dict[j['Movement ID']] = [] 
        for j in m_movements:
            movement_coor_dict[j['Movement ID']].append(j['Coordinates'])

        # calculates the overall trajectory length for each of the movement IDs   
        for j in movement_coor_dict:
            coor_list = movement_coor_dict[j]
            motion_start = False
            trajectory = 0
            if len(coor_list) > 1:
                trajectory_list = []
                if motion_start == True:
                    motion_start = False
                else:
                    count = 0
                    for k in coor_list:
                        trajectory_list.append(get_distance(coor_list[count-1], coor_list[count]))
                        count += 1
                    movement_id = j
                    trajectory = sum(trajectory_list)
                    single_coor = False
            else:
                movement_id = 1
                trajectory_list = [0]
                trajectory = 0
                single_coor = False
            m_df.loc[row_count] = [i, movement_id, trajectory, single_coor]
            row_count += 1
    m_df = m_df.sort_values(by=['test_number', 'movement_id'])
    
    for j in m_df['single_coor'].tolist():
        if j == True:
            m_df = movement_df.drop[count] 
        count += 1
    m_df = m_df.reset_index(drop=True)
    return m_df

In [5]:
all_true_k=[]
all_true_m=[]
all_false_k=[]
all_false_m=[]
for i in range(1, 89):
    user_number = i 
    user_number = str(user_number).zfill(4)
    f = open(r'../../../Dataset/behaviour_biometrics_dataset/raw_kmt_dataset/raw_kmt_user_' + user_number + '.json') # loads 1 of the 88 tests from drive
    data = json.load(f)
    user_details = data['details'] 
    true_data = data['true_data']
    false_data = data ['false_data']
    true_k_df = dict_key_conversion(true_data) 
    false_k_df = dict_key_conversion(false_data)

    true_m_df = dict_mouse_conversion(true_data) 
    false_m_df = dict_mouse_conversion(false_data)
    true_m_df['user']=i
    false_m_df['user']=i
    true_k_df['user']=i
    false_k_df['user']=i
    all_true_k.append(true_k_df)
    all_true_m.append(true_m_df)
    all_false_k.append(false_k_df)
    all_false_m.append(false_m_df)
    


In [6]:
import string

def key_mapping():
    standard_chars=list(string.digits)+list(string.ascii_lowercase)+list(string.ascii_uppercase)+list(string.punctuation)
    special_keys = [
        "shift", 
        "rshift", 
        "capslock", 
        "backspace", 
        "spacebar", 
        "tab", 
        "numpad0", "numpad1", "numpad2", "numpad3", "numpad4", 
        "numpad5", "numpad6", "numpad7", "numpad8", "numpad9",
        "numpadsubstract",
        "enter"
    ]
    all_keys = sorted(list(set(standard_chars+special_keys)))
    key_map = {key:i+1 for i, key in enumerate(all_keys)}

    return key_map

maped = key_mapping()

def get_id(key_str):
    return maped.get(str(key_str), 0)

In [7]:
full_true_k_df=pd.concat(all_true_k, ignore_index=True)
full_true_k_df['key_pressed']=full_true_k_df['key_pressed'].apply(get_id)


In [8]:
full_false_k_df=pd.concat(all_false_k, ignore_index=True)
full_false_k_df['key_pressed']=full_false_k_df['key_pressed'].apply(get_id)


In [9]:
dwell_times=full_true_k_df['dwell_time'].to_numpy()
dwell_mean=dwell_times.mean()
dwell_std=dwell_times.std()
flight_times=full_true_k_df['flight_time'].to_numpy()
flight_mean=flight_times.mean()
flight_std=flight_times.std()
def normalize(df):
    norm_df=df.copy()
    norm_df['dwell_time']=(df['dwell_time']-dwell_mean)/dwell_std
    norm_df['flight_time']=(df['flight_time']-flight_mean)/flight_std
    return norm_df
normalized_df=normalize(full_true_k_df)

In [10]:
normalized_df_f=normalize(full_false_k_df)

In [11]:

X_seq=[]
y_labels=[]
for u in range (1,89):
    main_data=normalized_df[normalized_df["user"]==u]
    for d in range(1,11):
        temp_data=[]
        temp_rows=main_data[main_data["test_number"]==d]
        temp_rows=temp_rows[["dwell_time", "flight_time", "key_pressed"]].values.tolist()   #.values will make it in 2d array form. so its like a list of 2d arrays
        X_seq.append(temp_rows);
        y_labels.append(u);
    
    

In [12]:
lengths = [len(seq) for seq in X_seq]
print(set(lengths))


{26, 28, 30, 32, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 89, 90, 93, 94, 95, 96, 97, 98, 101, 102, 109, 111, 112, 116, 121, 124, 127, 133, 143, 145, 156, 167, 187, 191, 199, 288}


In [13]:
from torch.nn.utils.rnn import pad_sequence
import torch
X_tensors=[torch.tensor(seq) for seq in X_seq]
X_padded=pad_sequence(X_tensors, batch_first=True, padding_value=0)


In [14]:
lengths = [len(seq) for seq in X_padded]
print(set(lengths))


{288}


In [15]:
from sklearn.model_selection import train_test_split
y_tensor=torch.tensor(y_labels)
x_train, x_test, y_train, y_test=train_test_split(X_padded, y_tensor, test_size=0.2)

In [16]:
import torch.nn as nn
import torch.optim as opt
import torch.nn.functional as F

In [17]:
class LSTM_keys(nn.Module):
    def __init__(self, key_dict_size):
        super(LSTM_keys, self).__init__()
        self.key_embed=nn.Embedding(key_dict_size+1, embedding_dim=8, padding_idx=0)   #didnt normalize keyids so we make embeddings
        self.lstm=nn.LSTM(input_size=10, hidden_size=64, num_layers=1, batch_first=True, dropout=0.0)
        self.hlayer1=nn.Linear(64, 32)
        self.relu=nn.ReLU()
        self.hlayer2=nn.Linear(32,16)
        #input=10 cuz 8 embeddings+flight+dwell, hidden size similar to memory and how uch the lstm remember and is outut embedding size

    def forward(self, x):
        dwell=x[:, :, 0].unsqueeze(-1)  # keeping unsqeeuze(-1) to add extra dimension cuz we adding keys to it
        flight=x[:, :, 1].unsqueeze(-1) 
        keys=x[:, :, 2].long()

        key_emb=self.key_embed(keys)
        
        ip=torch.cat([key_emb, dwell, flight], dim=2)
        op, (hidden, _)=self.lstm(ip)
        values=hidden[-1]

        x=self.hlayer2(self.relu(self.hlayer1(values)))

        return x

In [18]:
import torch.optim as opt


In [19]:
lstm1_model=LSTM_keys(key_dict_size=112)
temp_classifier=nn.Linear(16, 88)
optimizer=opt.AdamW(list(lstm1_model.parameters())+list(temp_classifier.parameters()), lr=0.005)
loss=nn.CrossEntropyLoss()
for epoch in range(100):
    lstm1_model.train()
    temp_classifier.train()
    optimizer.zero_grad()
    embedded=lstm1_model(x_train)
    classifier=temp_classifier(embedded)
    losses=loss(classifier, y_train-1)
    losses.backward()
    optimizer.step()
    print(f"Epoch {epoch} Loss={losses.item()} ")
    

Epoch 0 Loss=4.490215301513672 
Epoch 1 Loss=4.487006187438965 
Epoch 2 Loss=4.484399318695068 
Epoch 3 Loss=4.482227802276611 
Epoch 4 Loss=4.480238914489746 
Epoch 5 Loss=4.478360176086426 
Epoch 6 Loss=4.476655960083008 
Epoch 7 Loss=4.474978446960449 
Epoch 8 Loss=4.473633289337158 
Epoch 9 Loss=4.472537040710449 
Epoch 10 Loss=4.471613883972168 
Epoch 11 Loss=4.470836162567139 
Epoch 12 Loss=4.47015380859375 
Epoch 13 Loss=4.469552516937256 
Epoch 14 Loss=4.469057559967041 
Epoch 15 Loss=4.468672752380371 
Epoch 16 Loss=4.4683709144592285 
Epoch 17 Loss=4.468127727508545 
Epoch 18 Loss=4.4679179191589355 
Epoch 19 Loss=4.467709064483643 
Epoch 20 Loss=4.467470645904541 
Epoch 21 Loss=4.467184066772461 
Epoch 22 Loss=4.4668288230896 
Epoch 23 Loss=4.466365337371826 
Epoch 24 Loss=4.465770721435547 
Epoch 25 Loss=4.46503210067749 
Epoch 26 Loss=4.464162826538086 
Epoch 27 Loss=4.584754467010498 
Epoch 28 Loss=4.463573455810547 
Epoch 29 Loss=4.463908672332764 
Epoch 30 Loss=4.464185

In [20]:
lstm1_model.eval()
with torch.no_grad():
    emb_all = lstm1_model(x_train).numpy()  

In [21]:
target_uid = 1
user = np.array(y_labels)  
X = emb_all

In [22]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

lstm1_model.eval()
with torch.no_grad():
    emb_all = lstm1_model(X_padded).numpy()   
user_ids = (y_tensor.detach().numpy() + 1).astype(int)
unique_users = np.unique(user_ids)

results = []

for uid in unique_users:
    pos=(user_ids==uid)
    neg=(user_ids!=uid)

    X_pos=emb_all[pos]
    X_neg_all=emb_all[neg]
    rng = np.random.default_rng(42+uid)
    neg_idx=rng.choice(X_neg_all.shape[0], size=10, replace=False)
    X_neg=X_neg_all[neg_idx]
    n_neg=X_neg.shape[0]

    y_pos=np.ones(10, dtype=int)
    y_neg=np.zeros(10, dtype=int)

    X_user=np.vstack([X_pos, X_neg])
    y_user=np.concatenate([y_pos, y_neg])

    xi_train, xi_test, yi_train, yi_test=train_test_split(X_user, y_user, test_size=0.25, stratify=y_user, random_state=42)

    svm_clf = SVC(kernel="rbf",C=1.0,gamma="scale",class_weight="balanced")
    svm_clf.fit(xi_train,yi_train)
    y_train_pred = svm_clf.predict(xi_train)
    y_test_pred = svm_clf.predict(xi_test)
    train_acc = accuracy_score(yi_train, y_train_pred)
    test_acc = accuracy_score(yi_test, y_test_pred)
    results.append((uid, train_acc, test_acc))
    print(f"User={uid}, Training accuracy={train_acc}, tesitng accuracy={test_acc}")
mean_test=np.mean([r[2] for r in results])
print(f"\nNet accuracy is{ mean_test:.3f}")


User=2, Training accuracy=0.5333333333333333, tesitng accuracy=0.4
User=3, Training accuracy=0.5333333333333333, tesitng accuracy=0.4
User=4, Training accuracy=0.5333333333333333, tesitng accuracy=0.4
User=5, Training accuracy=0.5333333333333333, tesitng accuracy=0.4
User=6, Training accuracy=0.5333333333333333, tesitng accuracy=0.4
User=7, Training accuracy=0.5333333333333333, tesitng accuracy=0.4
User=8, Training accuracy=0.5333333333333333, tesitng accuracy=0.4
User=9, Training accuracy=0.5333333333333333, tesitng accuracy=0.4
User=10, Training accuracy=0.5333333333333333, tesitng accuracy=0.4
User=11, Training accuracy=0.5333333333333333, tesitng accuracy=0.4
User=12, Training accuracy=0.5333333333333333, tesitng accuracy=0.4
User=13, Training accuracy=0.5333333333333333, tesitng accuracy=0.4
User=14, Training accuracy=0.5333333333333333, tesitng accuracy=0.4
User=15, Training accuracy=0.5333333333333333, tesitng accuracy=0.4
User=16, Training accuracy=0.5333333333333333, tesitng a

In [23]:
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    roc_curve,
    roc_auc_score
)
import numpy as np

results = []
all_true_global = []
all_pred_global = []
all_score_global = []

for uid in unique_users:
    pos = (user_ids == uid)
    neg = (user_ids != uid)

    X_pos = emb_all[pos]
    X_neg_all = emb_all[neg]

    if len(X_pos) < 2:
        continue

    rng = np.random.default_rng(42 + uid)
    n_samples = min(len(X_pos), 10)

    neg_idx = rng.choice(X_neg_all.shape[0], size=n_samples, replace=False)
    X_neg = X_neg_all[neg_idx]

    y_pos = np.ones(n_samples, dtype=int)
    y_neg = np.zeros(n_samples, dtype=int)

    X_user = np.vstack([X_pos, X_neg])
    y_user = np.concatenate([y_pos, y_neg])

    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    user_true = []
    user_pred = []
    user_scores = []

    for train_idx, test_idx in kfold.split(X_user, y_user):
        X_train, X_test = X_user[train_idx], X_user[test_idx]
        y_train, y_test = y_user[train_idx], y_user[test_idx]

        clf = SVC(kernel="rbf", C=1.0, gamma="scale")
        clf.fit(X_train, y_train)

        preds = clf.predict(X_test)
        scores = clf.decision_function(X_test)

        user_true.extend(y_test)
        user_pred.extend(preds)
        user_scores.extend(scores)

    all_true_global.extend(user_true)
    all_pred_global.extend(user_pred)
    all_score_global.extend(user_scores)

    acc = accuracy_score(user_true, user_pred)
    print(f"User={uid}, Avg CV Accuracy={acc:.3f}")
    results.append(acc)

cm = confusion_matrix(all_true_global, all_pred_global)
TN, FP, FN, TP = cm.ravel()

# Metrics
final_accuracy = accuracy_score(all_true_global, all_pred_global)
final_f1 = f1_score(all_true_global, all_pred_global)
FAR = FP / (FP + TN)

# ROC
fpr, tpr, thresholds = roc_curve(all_true_global, all_score_global)
roc_auc = roc_auc_score(all_true_global, all_score_global)

print(f"Accuracy : {final_accuracy:.4f}")
print(f"F1 Score: {final_f1:.4f}")
print(f"FAR: {FAR:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")
print("Confusion Matrix:")
print(cm)


User=2, Avg CV Accuracy=0.500
User=3, Avg CV Accuracy=0.600
User=4, Avg CV Accuracy=0.600
User=5, Avg CV Accuracy=0.450
User=6, Avg CV Accuracy=0.500
User=7, Avg CV Accuracy=0.700
User=8, Avg CV Accuracy=0.450
User=9, Avg CV Accuracy=0.600
User=10, Avg CV Accuracy=0.450
User=11, Avg CV Accuracy=0.550
User=12, Avg CV Accuracy=0.350
User=13, Avg CV Accuracy=0.550
User=14, Avg CV Accuracy=0.550
User=15, Avg CV Accuracy=0.650
User=16, Avg CV Accuracy=0.650
User=17, Avg CV Accuracy=0.550
User=18, Avg CV Accuracy=0.400
User=19, Avg CV Accuracy=0.500
User=20, Avg CV Accuracy=0.450
User=21, Avg CV Accuracy=0.650
User=22, Avg CV Accuracy=0.400
User=23, Avg CV Accuracy=0.400
User=24, Avg CV Accuracy=0.450
User=25, Avg CV Accuracy=0.550
User=26, Avg CV Accuracy=0.500
User=27, Avg CV Accuracy=0.650
User=28, Avg CV Accuracy=0.600
User=29, Avg CV Accuracy=0.450
User=30, Avg CV Accuracy=0.450
User=31, Avg CV Accuracy=0.600
User=32, Avg CV Accuracy=0.600
User=33, Avg CV Accuracy=0.600
User=34, Avg CV 

In [29]:
#####NEW METHOD INSTEAD OF TRAIN TEST SPLIT ITS KFOLD CROSS VALIDATION #####


from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
results = []

for uid in unique_users:
    # 1. Prepare Data
    pos = (user_ids == uid)
    neg = (user_ids != uid)
    
    X_pos = emb_all[pos]
    X_neg_all = emb_all[neg]
    
    # Safety Check
    if len(X_pos) < 2: continue

    # 2. Select Negatives
    rng = np.random.default_rng(42 + uid)
    # Ensure we don't crash if X_pos has fewer than 10 items
    n_samples = min(len(X_pos), 10) 
    
    neg_idx = rng.choice(X_neg_all.shape[0], size=n_samples, replace=False)
    X_neg = X_neg_all[neg_idx]

    # 3. Create Dataset
    y_pos = np.ones(n_samples, dtype=int)
    y_neg = np.zeros(n_samples, dtype=int)

    X_user = np.vstack([X_pos, X_neg])
    y_user = np.concatenate([y_pos, y_neg])

    # 4. CROSS VALIDATION (The Fix)
    # We split the 20 samples into 5 chunks. 
    # We train on 16, test on 4. Repeat 5 times.
    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    scores = []
    
    for train_index, test_index in kfold.split(X_user, y_user):
        X_train_k, X_test_k = X_user[train_index], X_user[test_index]
        y_train_k, y_test_k = y_user[train_index], y_user[test_index]
        
        clf = SVC(kernel="rbf", C=1.0, gamma="scale")
        clf.fit(X_train_k, y_train_k)
        
        pred = clf.predict(X_test_k)
        scores.append(accuracy_score(y_test_k, pred))
    
    # Average the 5 scores
    avg_score = np.mean(scores)
    
    results.append((uid, avg_score))
    print(f"User={uid}, Avg Test Acc={avg_score:.3f}")

# Final Calculation
final_avg = np.mean([r[1] for r in results])
print(f"\nFINAL SYSTEM ACCURACY: {final_avg:.3f}")



User=2, Avg Test Acc=0.700
User=3, Avg Test Acc=0.450
User=4, Avg Test Acc=0.600
User=5, Avg Test Acc=0.500
User=6, Avg Test Acc=0.550
User=7, Avg Test Acc=0.550
User=8, Avg Test Acc=0.450
User=9, Avg Test Acc=0.650
User=10, Avg Test Acc=0.450
User=11, Avg Test Acc=0.700
User=12, Avg Test Acc=0.500
User=13, Avg Test Acc=0.400
User=14, Avg Test Acc=0.600
User=15, Avg Test Acc=0.550
User=16, Avg Test Acc=0.700
User=17, Avg Test Acc=0.650
User=18, Avg Test Acc=0.450
User=19, Avg Test Acc=0.600
User=20, Avg Test Acc=0.400
User=21, Avg Test Acc=0.700
User=22, Avg Test Acc=0.600
User=23, Avg Test Acc=0.450
User=24, Avg Test Acc=0.500
User=25, Avg Test Acc=0.500
User=26, Avg Test Acc=0.650
User=27, Avg Test Acc=0.650
User=28, Avg Test Acc=0.350
User=29, Avg Test Acc=0.550
User=30, Avg Test Acc=0.700
User=31, Avg Test Acc=0.600
User=32, Avg Test Acc=0.500
User=33, Avg Test Acc=0.650
User=34, Avg Test Acc=0.700
User=35, Avg Test Acc=0.600
User=36, Avg Test Acc=0.550
User=37, Avg Test Acc=0.650
